In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from PROPS_EV.calculateEVS import *
from BACKTEST.backtest import *
from MODELS.pipeline import *

### Load Model

In [2]:
PTSmodel = joblib.load('Models/xgbPTSModel.pkl')
PTSfeatures = joblib.load('Models/topPTSfeatures.pkl')
REBmodel = joblib.load('Models/xgbREBModel.pkl')
REBfeatures = joblib.load('Models/topREBfeatures.pkl')

### Load Data

In [5]:
pd.set_option('display.max_columns', None)


s25_pts = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
s24_pts = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_24.csv')
# s25_reb = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/REB_TRAIN_25.csv')
# s24_reb = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/REB_TRAIN_24.csv')

dfPTS = pd.concat([s25_pts, s24_pts]).sort_values(by='GAME_DATE')
# dfREB = pd.concat([s25_reb, s24_reb]).sort_values(by='GAME_DATE')


usData = pd.read_csv('../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_20251014.csv')
singlePTSBookies = usData[usData['CATEGORY'] == 'player_points']


dfsData = pd.read_csv('../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_20251019.csv')
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points')]
# dfsREB = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_rebounds')]
dfsPTS.rename(columns={'OVER/UNDER':'SIDE'}, inplace=True)
dfsPTS
# dfsREB.rename(columns={'OVER/UNDER':'SIDE'}, inplace=True)

C:\Users\alexg\AppData\Local\Temp\ipykernel_18376\3033148287.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dfsPTS.rename(columns={'OVER/UNDER':'SIDE'}, inplace=True)


,BOOKMAKER,CATEGORY,NAME,SIDE,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,PrizePicks,player_points,Luguentz Dort,Over,10.0,-137,2025-10-21,2025-10-20T03:05:29Z
1,PrizePicks,player_points,Luguentz Dort,Under,10.0,-137,2025-10-21,2025-10-20T03:05:29Z
2,PrizePicks,player_points,Alperen Sengun,Over,18.5,-137,2025-10-21,2025-10-20T03:05:29Z
3,PrizePicks,player_points,Alperen Sengun,Under,18.5,-137,2025-10-21,2025-10-20T03:05:29Z
4,PrizePicks,player_points,Jabari Smith Jr,Over,12.0,-137,2025-10-21,2025-10-20T03:05:29Z
...,...,...,...,...,...,...,...,...
286,PrizePicks,player_points,Dario Saric,Under,8.5,-137,2025-10-23,2025-10-20T03:10:32Z
287,PrizePicks,player_points,Drew Eubanks,Over,8.5,-137,2025-10-23,2025-10-20T03:10:32Z
288,PrizePicks,player_points,Drew Eubanks,Under,8.5,-137,2025-10-23,2025-10-20T03:10:32Z
289,PrizePicks,player_points,Mark Williams,Over,12.5,-137,2025-10-23,2025-10-20T03:10:32Z


### Top EVs for single bets

In [ ]:
results = single_bet(
    data=dfPTS,
    bookmakers=singlePTSBookies,
    model=PTSmodel,
    features=PTSfeatures,
    current_date='2025-10-21',
    edge_threshold=4.5,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
results.sort_values(by='EV%', ascending=False).head(10)

In [ ]:
results = single_bet(
    data=dfREB,
    bookmakers=singleREBBookies,
    model=REBmodel,
    features=REBfeatures,
    edge_threshold=2.0,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=1.5,
    max_std=6.5,
    stat_col='REB'
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
412,Thomas Bryant,betmgm,rebounds,10.5,125,under,5.142313,1,0.006,0.994,0.444,123.74,0.99,0.49,0.25,"(1.4, 9.2)"
1531,Kris Murray,betmgm,rebounds,4.5,135,under,1.732772,1,0.101,0.899,0.426,111.24,0.82,0.41,0.21,"(0.1, 6.0)"
1448,Donovan Clingan,espnbet,rebounds,9.5,105,under,3.203625,1,0.002,0.998,0.488,104.61,1.00,0.50,0.25,"(0.3, 7.4)"
1555,Zeke Nnaji,espnbet,rebounds,5.5,115,under,2.452899,1,0.069,0.931,0.465,100.12,0.87,0.44,0.22,"(0.2, 6.4)"
1726,Olivier-Maxence Prosper,fanduel,rebounds,8.5,100,under,1.889173,1,0.001,0.999,0.500,99.88,1.00,0.50,0.25,"(0.2, 6.0)"
1557,Zeke Nnaji,fanduel,rebounds,5.5,112,under,2.452899,1,0.073,0.927,0.472,96.59,0.86,0.43,0.22,"(0.2, 6.5)"
1450,Donovan Clingan,betmgm,rebounds,9.5,-105,under,3.203625,1,0.002,0.998,0.512,94.89,1.00,0.50,0.25,"(0.3, 7.5)"
229,Keldon Johnson,draftkings,rebounds,2.5,150,over,4.210320,0,0.777,0.223,0.400,94.25,0.63,0.31,0.16,"(0.5, 9.7)"
1156,Bobby Portis,betmgm,rebounds,6.5,115,over,9.395657,1,0.903,0.097,0.465,94.10,0.82,0.41,0.20,"(5.0, 13.7)"
409,Thomas Bryant,espnbet,rebounds,8.5,105,under,5.142313,1,0.053,0.947,0.488,94.07,0.90,0.45,0.22,"(1.3, 9.2)"


### Top EVs for 2 leg bets

In [4]:
results = prizepickspairsEV(
    data=dfPTS,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    current_date='2025-10-21',
    edge_threshold=5,
    stake=100,
    simulations=1000,
    std_window=15,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Jarrett Allen,player_points,Underdog,-137,12.5,2.91,UNDER,0.050,0.950,"(0.0, 12.6)",Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(1.5, 21.6)",UNDER/UNDER,1,0.9025,1.708,0.854
1,Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(1.5, 21.6)",Zion Williamson,player_points,Underdog,-137,24.5,15.14,UNDER,0.050,0.950,"(6.3, 23.7)",UNDER/UNDER,1,0.9025,1.708,0.854
2,Jarrett Allen,player_points,Underdog,-137,12.5,2.91,UNDER,0.050,0.950,"(0.0, 12.6)",Zion Williamson,player_points,Underdog,-137,24.5,15.14,UNDER,0.050,0.950,"(6.3, 23.7)",UNDER/UNDER,1,0.9025,1.708,0.854
3,Jarrett Allen,player_points,Underdog,-137,12.5,2.91,UNDER,0.050,0.950,"(0.0, 12.6)",Trey Murphy III,player_points,Underdog,-137,19.5,1.83,UNDER,0.050,0.950,"(0.0, 15.1)",UNDER/UNDER,1,0.9025,1.708,0.854
4,Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(1.5, 21.6)",Trey Murphy III,player_points,Underdog,-137,19.5,1.83,UNDER,0.050,0.950,"(0.0, 15.1)",UNDER/UNDER,1,0.9025,1.708,0.854
5,Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(1.5, 21.6)",Matas Buzelis,player_points,Underdog,-137,14.5,8.43,UNDER,0.057,0.943,"(0.0, 16.0)",UNDER/UNDER,1,0.8958,1.688,0.844
6,Matas Buzelis,player_points,Underdog,-137,14.5,8.43,UNDER,0.057,0.943,"(0.0, 16.0)",Trey Murphy III,player_points,Underdog,-137,19.5,1.83,UNDER,0.050,0.950,"(0.0, 15.1)",UNDER/UNDER,1,0.8958,1.688,0.844
7,Jarrett Allen,player_points,Underdog,-137,12.5,2.91,UNDER,0.050,0.950,"(0.0, 12.6)",Matas Buzelis,player_points,Underdog,-137,14.5,8.43,UNDER,0.057,0.943,"(0.0, 16.0)",UNDER/UNDER,1,0.8958,1.688,0.844
8,Matas Buzelis,player_points,Underdog,-137,14.5,8.43,UNDER,0.057,0.943,"(0.0, 16.0)",Zion Williamson,player_points,Underdog,-137,24.5,15.14,UNDER,0.050,0.950,"(6.3, 23.7)",UNDER/UNDER,1,0.8958,1.688,0.844
9,Victor Wembanyama,player_points,Underdog,-137,24.5,12.70,UNDER,0.071,0.929,"(0.0, 28.3)",Zion Williamson,player_points,Underdog,-137,24.5,15.14,UNDER,0.050,0.950,"(6.3, 23.7)",UNDER/UNDER,1,0.8826,1.648,0.824


In [ ]:
results = prizepickspairsEV(
    data=dfREB,
    bookmakers=dfsREB,
    model=REBmodel,
    features=REBfeatures,
    edge_threshold=2.0,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=1.5,
    max_std=6.5,
    stat_col='REB'
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Immanuel Quickley,player_rebounds,PrizePicks,-137,3.5,5.32,OVER,0.779,0.221,"(0.5, 10.0)",Jalen Johnson,player_rebounds,PrizePicks,-137,9.5,6.81,UNDER,0.176,0.824,"(1.3, 12.3)",OVER/UNDER,0,0.6417,0.925,0.463
1,De'Andre Hunter,player_rebounds,PrizePicks,-137,3.5,4.80,OVER,0.738,0.262,"(0.9, 8.7)",Jalen Johnson,player_rebounds,PrizePicks,-137,9.5,6.81,UNDER,0.176,0.824,"(1.3, 12.3)",OVER/UNDER,0,0.6083,0.825,0.412
2,Jalen Johnson,player_rebounds,PrizePicks,-137,9.5,6.81,UNDER,0.176,0.824,"(1.3, 12.3)",Klay Thompson,player_rebounds,PrizePicks,-137,2.5,3.73,OVER,0.730,0.270,"(0.0, 7.7)",UNDER/OVER,0,0.6015,0.805,0.402
3,Jalen Johnson,player_rebounds,PrizePicks,-137,9.5,6.81,UNDER,0.176,0.824,"(1.3, 12.3)",Stephon Castle,player_rebounds,PrizePicks,-137,4.5,5.67,OVER,0.723,0.277,"(1.8, 9.6)",UNDER/OVER,0,0.5960,0.788,0.394
4,Jalen Johnson,player_rebounds,PrizePicks,-137,9.5,6.81,UNDER,0.176,0.824,"(1.3, 12.3)",Jalen Williams,player_rebounds,PrizePicks,-137,5.0,6.05,OVER,0.706,0.294,"(2.0, 10.2)",UNDER/OVER,0,0.5816,0.745,0.372
5,De'Andre Hunter,player_rebounds,PrizePicks,-137,3.5,4.80,OVER,0.738,0.262,"(0.9, 8.7)",Immanuel Quickley,player_rebounds,PrizePicks,-137,3.5,5.32,OVER,0.779,0.221,"(0.5, 10.0)",OVER/OVER,0,0.5746,0.724,0.362
6,Immanuel Quickley,player_rebounds,PrizePicks,-137,3.5,5.32,OVER,0.779,0.221,"(0.5, 10.0)",Klay Thompson,player_rebounds,PrizePicks,-137,2.5,3.73,OVER,0.730,0.270,"(0.0, 7.7)",OVER/OVER,0,0.5682,0.705,0.352
7,Jalen Johnson,player_rebounds,PrizePicks,-137,9.5,6.81,UNDER,0.176,0.824,"(1.3, 12.3)",Mitchell Robinson,player_rebounds,PrizePicks,-137,7.0,5.12,UNDER,0.312,0.688,"(0.0, 13.1)",UNDER/UNDER,0,0.5673,0.702,0.351
8,Immanuel Quickley,player_rebounds,PrizePicks,-137,3.5,5.32,OVER,0.779,0.221,"(0.5, 10.0)",Stephon Castle,player_rebounds,PrizePicks,-137,4.5,5.67,OVER,0.723,0.277,"(1.8, 9.6)",OVER/OVER,0,0.5630,0.689,0.345
9,Immanuel Quickley,player_rebounds,PrizePicks,-137,3.5,5.32,OVER,0.779,0.221,"(0.5, 10.0)",Jalen Williams,player_rebounds,PrizePicks,-137,5.0,6.05,OVER,0.706,0.294,"(2.0, 10.2)",OVER/OVER,0,0.5495,0.648,0.324


## 3 leg parlay

In [5]:
threeLeg = prizepicks3LegEV(
    data=dfPTS,
    bookmakers=dfsPTS,
    model=PTSmodel,
    features=PTSfeatures,
    current_date='2025-10-21',
    edge_threshold=4.5,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0,
    stat_col='PTS'
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)

Processing 3-leg parlays...
Note: PHX plays on 2025-10-22 (not 2025-10-21)
Note: PHX plays on 2025-10-22 (not 2025-10-21)
Note: PHX plays on 2025-10-22 (not 2025-10-21)


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,PLAYER 3,CATEGORY 3,BOOKMAKER 3,ODDS 3,LINE 3,PREDICTION 3,MODEL_SIDE 3,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
5,Alperen Sengun,player_points,Underdog,-137,18.5,13.64,UNDER,0.103,0.897,"(6.2, 21.2)",Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(0.0, 23.3)",Shai Gilgeous-Alexander,player_points,Underdog,-137,31.5,19.26,UNDER,0.102,0.898,"(0.0, 38.0)",UNDER/UNDER/UNDER,1,0.7657,3.594,0.719
17,Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(0.0, 23.3)",Shai Gilgeous-Alexander,player_points,Underdog,-137,31.5,19.26,UNDER,0.102,0.898,"(0.0, 38.0)",Steven Adams,player_points,Underdog,-137,5.5,7.57,OVER,0.841,0.159,"(3.6, 11.6)",UNDER/UNDER/OVER,0,0.7174,3.304,0.661
6,Alperen Sengun,player_points,Underdog,-137,18.5,13.64,UNDER,0.103,0.897,"(6.2, 21.2)",Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(0.0, 23.3)",Steven Adams,player_points,Underdog,-137,5.5,7.57,OVER,0.841,0.159,"(3.6, 11.6)",UNDER/UNDER/OVER,0,0.7166,3.300,0.660
9,Alperen Sengun,player_points,Underdog,-137,18.5,13.64,UNDER,0.103,0.897,"(6.2, 21.2)",Shai Gilgeous-Alexander,player_points,Underdog,-137,31.5,19.26,UNDER,0.102,0.898,"(0.0, 38.0)",Steven Adams,player_points,Underdog,-137,5.5,7.57,OVER,0.841,0.159,"(3.6, 11.6)",UNDER/UNDER/OVER,0,0.6776,3.065,0.613
11,Isaiah Hartenstein,player_points,Underdog,-137,10.5,9.00,UNDER,0.362,0.638,"(1.1, 17.2)",Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(0.0, 23.3)",Shai Gilgeous-Alexander,player_points,Underdog,-137,31.5,19.26,UNDER,0.102,0.898,"(0.0, 38.0)",UNDER/UNDER/UNDER,0,0.5441,2.264,0.453
0,Alperen Sengun,player_points,Underdog,-137,18.5,13.64,UNDER,0.103,0.897,"(6.2, 21.2)",Isaiah Hartenstein,player_points,Underdog,-137,10.5,9.00,UNDER,0.362,0.638,"(1.1, 17.2)",Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(0.0, 23.3)",UNDER/UNDER/UNDER,0,0.5435,2.261,0.452
2,Alperen Sengun,player_points,Underdog,-137,18.5,13.64,UNDER,0.103,0.897,"(6.2, 21.2)",Isaiah Hartenstein,player_points,Underdog,-137,10.5,9.00,UNDER,0.362,0.638,"(1.1, 17.2)",Shai Gilgeous-Alexander,player_points,Underdog,-137,31.5,19.26,UNDER,0.102,0.898,"(0.0, 38.0)",UNDER/UNDER/UNDER,0,0.5139,2.083,0.417
12,Isaiah Hartenstein,player_points,Underdog,-137,10.5,9.00,UNDER,0.362,0.638,"(1.1, 17.2)",Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(0.0, 23.3)",Steven Adams,player_points,Underdog,-137,5.5,7.57,OVER,0.841,0.159,"(3.6, 11.6)",UNDER/UNDER/OVER,0,0.5092,2.055,0.411
15,Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(0.0, 23.3)",Luguentz Dort,player_points,Underdog,-137,10.5,8.62,UNDER,0.406,0.594,"(0.0, 23.9)",Shai Gilgeous-Alexander,player_points,Underdog,-137,31.5,19.26,UNDER,0.102,0.898,"(0.0, 38.0)",UNDER/UNDER/UNDER,0,0.5069,2.041,0.408
4,Alperen Sengun,player_points,Underdog,-137,18.5,13.64,UNDER,0.103,0.897,"(6.2, 21.2)",Kevin Durant,player_points,Underdog,-137,25.5,11.60,UNDER,0.050,0.950,"(0.0, 23.3)",Luguentz Dort,player_points,Underdog,-137,10.5,8.62,UNDER,0.406,0.594,"(0.0, 23.9)",UNDER/UNDER/UNDER,0,0.5063,2.038,0.408


In [ ]:
threeLeg = prizepicks3LegEV(
    data=dfREB,
    bookmakers=dfsREB,
    model=REBmodel,
    features=REBfeatures,
    edge_threshold=2.0,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=1.5,
    max_std=6.5,
    stat_col='REB'
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)

Processing 3-leg parlays...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,PLAYER 3,CATEGORY 3,BOOKMAKER 3,ODDS 3,LINE 3,PREDICTION 3,MODEL_SIDE 3,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
7201,Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4678,1.807,0.361
8216,Pascal Siakam,player_rebounds,prizepicks,-137,7.5,5.85,UNDER,0.256,0.744,"(1.3, 10.7)",Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4596,1.757,0.351
8315,Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Thomas Bryant,player_rebounds,prizepicks,-137,6.5,5.14,UNDER,0.259,0.741,"(1.3, 9.1)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4576,1.746,0.349
6055,Jalen Duren,player_rebounds,prizepicks,-137,11.5,14.22,OVER,0.739,0.261,"(5.7, 22.8)",Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",OVER/UNDER/OVER,0,0.4568,1.741,0.348
7177,Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Pascal Siakam,player_rebounds,prizepicks,-137,7.5,5.85,UNDER,0.256,0.744,"(1.3, 10.7)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4543,1.726,0.345
5084,Immanuel Quickley,player_rebounds,prizepicks,-137,3.5,4.89,OVER,0.735,0.265,"(0.8, 9.6)",Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",OVER/UNDER/OVER,0,0.4541,1.724,0.345
8306,Precious Achiuwa,player_rebounds,prizepicks,-137,8.5,7.05,UNDER,0.234,0.766,"(3.1, 10.9)",Santi Aldama,player_rebounds,prizepicks,-137,5.5,7.50,OVER,0.733,0.267,"(1.6, 14.3)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/OVER/OVER,0,0.4531,1.719,0.344
7216,Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Thomas Bryant,player_rebounds,prizepicks,-137,6.5,5.14,UNDER,0.259,0.741,"(1.3, 9.1)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",UNDER/UNDER/OVER,0,0.4524,1.714,0.343
5887,Jalen Duren,player_rebounds,prizepicks,-137,11.5,14.22,OVER,0.739,0.261,"(5.7, 22.8)",Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",OVER/UNDER/OVER,0,0.4516,1.710,0.342
4916,Immanuel Quickley,player_rebounds,prizepicks,-137,3.5,4.89,OVER,0.735,0.265,"(0.8, 9.6)",Josh Hart,player_rebounds,prizepicks,-137,9.5,7.59,UNDER,0.243,0.757,"(2.4, 12.9)",Zach Edey,player_rebounds,prizepicks,-137,7.0,9.92,OVER,0.807,0.193,"(3.2, 16.6)",OVER/UNDER/OVER,0,0.4488,1.693,0.339
